# Volatility Harvesting Triplet Scanner (High-Vol Only)

This notebook:
1. Loads a CSV of ETF tickers
2. Tests **triplet combinations** for a **high-volatility-only** volatility harvesting strategy
3. Ranks triplets by simple risk-adjusted metrics

**Key idea:** The strategy is *only active when realized volatility is high* (no trend filters).


## Setup

In [1]:
import itertools
from typing import Tuple, Optional
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Load tickers

In [2]:
import pandas as pd, numpy as np, re
from itertools import product

CSV_PATH = "../../etf-list.csv"

df = pd.read_csv(CSV_PATH)
df["Ticker"] = df["Ticker"].astype(str).str.upper().str.strip()
df["Category"] = df["Category"].astype(str).str.strip()
df["Subcategory"] = df["Subcategory"].astype(str).str.strip().replace({"nan": np.nan})

# 1) Exclude obvious non-core groups
EXCLUDE_CATS = {"Leveraged/Inverse", "Crypto", "Option Strategy", "Other"}
df = df[~df["Category"].isin(EXCLUDE_CATS)].copy()

# 2) Extra filter: remove leveraged/inverse by name heuristics (optional but recommended)
name = df["Fund Name"].astype(str).str.lower()
lev_pat = r"(direxion|ultra|3x|2x|1\.5x|bear|bull|leveraged|inverse)"
df = df[~name.str.contains(lev_pat, regex=True, na=False)].copy()



# -----------------------------
# Checkpoint config
# -----------------------------
CHECKPOINT_PATH = "triplet_scan_results.csv"   # append-only results
FLUSH_EVERY = 25                          # write to disk every N successes
PRINT_EVERY = 500                         # progress prints

# Load existing checkpoint (if any)
if os.path.exists(CHECKPOINT_PATH):
    done_df = pd.read_csv(CHECKPOINT_PATH)
    done_keys = set(done_df["Triplet"].astype(str))
    rows = done_df.to_dict("records")  # resume rows already computed
    print(f"Resuming from checkpoint: {len(done_keys)} completed")
else:
    done_keys = set()
    rows = []
    print("No checkpoint found. Starting fresh.")

# 3) Map to macro buckets
def to_bucket(row):
    cat = row["Category"]
    if cat in {"Bond", "Loan"}:
        return "bond"
    if cat == "REIT":
        return "reit"
    if cat == "Commodity":
        return "commodity"
    if cat in {"Short", "Currency"}:
        return "cash_fx"
    if cat in {"Index","Equity Sector","Factor","Dividend/Income","Country/Region","Thematic/Innovation","Growth","Global/World"}:
        return "equity"
    return "other"

df["bucket"] = df.apply(to_bucket, axis=1)

# 4) Pick top-N by Avg-Vol (liquidity proxy)
def top_by_vol(df_, n):
    x = df_.copy()
    x["Avg-Vol"] = pd.to_numeric(x["Avg-Vol"], errors="coerce")
    x = x.sort_values("Avg-Vol", ascending=False)
    return x.head(n)

TOP_EQUITY = 60
TOP_BOND = 30
TOP_COMMODITY = 15
TOP_REIT = 5

eq = top_by_vol(df[df["bucket"]=="equity"], TOP_EQUITY)["Ticker"].unique().tolist()
bd = top_by_vol(df[df["bucket"]=="bond"], TOP_BOND)["Ticker"].unique().tolist()
cm = top_by_vol(df[df["bucket"]=="commodity"], TOP_COMMODITY)["Ticker"].unique().tolist()
rt = top_by_vol(df[df["bucket"]=="reit"], TOP_REIT)["Ticker"].unique().tolist()

# 5) Build triplets by template
triplets = []
triplets += [{"A": a, "B": b, "C": c, "template": "equity_bond_commodity"} for a,b,c in product(eq, bd, cm)]
triplets += [{"A": a, "B": b, "C": c, "template": "equity_bond_reit"}      for a,b,c in product(eq, bd, rt)]

triplets_df = pd.DataFrame(triplets)
print("Triplets:", len(triplets_df))
triplets_df.head()


/var/folders/nf/0f78fmr95hz6c9cwnjjzzdlr0000gp/T/ipykernel_4222/1087224855.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~name.str.contains(lev_pat, regex=True, na=False)].copy()


Resuming from checkpoint: 21150 completed
Triplets: 36000


,A,B,C,template
0,SPY,TLT,SLV,equity_bond_commodity
1,SPY,TLT,GDX,equity_bond_commodity
2,SPY,TLT,GLD,equity_bond_commodity
3,SPY,TLT,USO,equity_bond_commodity
4,SPY,TLT,UNG,equity_bond_commodity


## Fetch prices

In [3]:
# -----------------------------
# 2) Fetch prices (Close with auto_adjust=True)
# -----------------------------
START = "2010-01-01"
END = None  # or "2025-12-31"
AUTO_ADJUST = True

def fetch_prices_yf(tickers, start=START, end=END, auto_adjust=AUTO_ADJUST) -> pd.DataFrame:
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=auto_adjust,
        progress=False,
        group_by="column",
        threads=True,
    )
    if isinstance(data.columns, pd.MultiIndex):
        # With group_by="column", top level is price field.
        close = data["Close"] if "Close" in data.columns.get_level_values(0) else data
        px = close.copy()
    else:
        px = data[["Close"]].rename(columns={"Close": tickers[0]})
    px = px.dropna(how="all")
    px.columns = [c.upper() for c in px.columns]
    return px

px_all = fetch_prices_yf(df["Ticker"].to_list())
print("Price panel shape:", px_all.shape)
px_all.tail()


Price panel shape: (4048, 228)


,AAAU,ACWI,AGG,AMLP,ARKK,ASHR,BABX,BBJP,BIL,BINC,BIV,BKLN,BND,BNDX,BSV,CALF,CGCP,CGDV,CGGR,CGUS,CIBR,COPX,COWZ,DFAC,DFAI,DFIV,DFSV,DGRO,DIA,DOG,DYNF,EEM,EFA,EFV,EMB,EMLC,EMXC,EWA,EWC,EWG,EWH,EWJ,EWT,EWU,EWW,EWY,EWZ,EZU,FBND,FENY,FEZ,FLOT,FNDF,FNDX,FPE,FXI,GDX,GDXJ,GLD,GLDM,GOVT,HYG,HYMB,IAU,IAUM,IBB,ICLN,IDEV,IEF,IEFA,IEI,IEMG,IGIB,IGSB,IGV,IJH,IJR,ILF,INDA,IQLT,ITB,ITOT,IUSB,IVV,IVW,IWD,IWF,IWM,IWN,IWR,IXUS,IYR,JAAA,JEPI,JEPQ,JETS,JNK,KBE,KBWB,KRE,...,SCHK,SCHM,SCHO,SCHP,SCHR,SCHV,SCHZ,SCZ,SDVY,SGOL,SGOV,SH,SHLD,SHV,SHY,SHYG,SIL,SILJ,SIVR,SJNK,SLV,SMH,SOXX,SPAB,SPDW,SPEM,SPHQ,SPHY,SPIB,SPLB,SPLG,SPLV,SPMD,SPMO,SPSB,SPSM,SPTI,SPTL,SPTS,SPY,SPYD,SPYG,SPYI,SPYV,SRLN,SVIX,SVXY,TBIL,TFLO,TIP,TLH,TLT,TSLR,UNG,URA,USFR,USHY,USIG,USMV,USO,VCIT,VCLT,VCSH,VEA,VEU,VFLO,VGIT,VGK,VGLT,VGSH,VIXY,VMBS,VNQ,VOO,VT,VTEB,VTI,VTIP,VTV,VTWO,VWO,VXUS,VXX,XBI,XHB,XLB,XLC,XLE,XLF,XLG,XLI,XLK,XLP,XLRE,XLU,XLV,XLY,XME,XOP,XRT
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-01-30,47.860001,145.500000,99.805000,50.029999,74.870003,33.439999,46.759998,69.910004,91.375999,52.907001,77.596001,20.740000,73.985001,48.458000,78.664001,45.380001,22.639999,44.869999,44.459999,40.950001,69.059998,84.809998,62.270000,40.680000,40.040001,52.900002,35.130001,71.830002,489.029999,23.160000,61.509998,59.099998,100.739998,75.339996,96.165001,26.178999,79.820000,27.750000,54.259998,43.110001,23.379999,85.720001,68.059998,46.160000,75.120003,122.410004,37.040001,66.660004,46.020000,28.280001,66.800003,50.866001,48.560001,28.379999,18.299999,39.610001,94.199997,124.089996,444.950012,96.010002,22.971001,80.721001,25.004000,91.199997,48.480000,172.429993,18.150000,86.309998,95.628006,94.050003,118.982994,72.559998,53.800999,52.912003,90.309998,68.669998,127.029999,35.360001,51.740002,47.610001,102.029999,150.979996,46.485001,695.030029,123.889999,219.860001,465.950012,259.649994,193.880005,99.180000,89.410004,96.209999,50.669998,58.216000,58.964001,27.950001,97.260002,63.570000,86.199997,68.809998,...,33.259998,31.620001,24.333000,26.639999,24.982000,31.110001,23.365000,81.769997,40.386002,46.250000,100.360992,35.680000,75.239998,110.098000,82.739998,42.850998,94.300003,31.799999,79.199997,25.289999,75.440002,403.459991,346.299988,25.737999,46.980000,49.200001,77.389999,23.681999,33.788998,22.597000,81.400002,73.650002,60.250000,119.870003,30.202000,49.459999,28.737000,26.347000,29.227999,691.969971,45.279999,107.250000,52.709999,58.169998,40.711998,22.700001,54.099998,49.880001,50.479000,110.480003,101.220001,86.797997,28.730000,16.90,54.990002,50.349998,37.424999,51.765999,94.989998,79.519997,83.619995,75.863998,79.752998,66.199997,77.730003,39.470001,59.709000,87.500000,55.514999,58.674999,26.709999,47.075001,90.800003,636.219971,145.449997,50.420002,340.570007,49.730000,199.750000,104.919998,56.470001,79.660004,27.500000,124.750000,108.400002,49.270000,120.080002,51.049999,53.439999,59.009998,165.440002,143.880005,83.510002,41.430000,43.250000,154.740005,121.169998,118.720001,140.240005,86.330002
2026-02-02,45.930000,146.309998,99.690002,49.590000,74.360001,33.080002,46.105999,69.970001,91.389999,52.900002,77.500000,20.770000,73.910004,48.430000,78.620003,45.820000,22.639999,44.939999,44.540001,41.110001,69.589996,85.199997,62.494999,40.970001,40.270000,53.230000,35.549999,72.330002,494.029999,22.930000,61.930000,59.279999,101.400002,75.949997,96.139999,26.209999,80.389999,27.950001,54.549999,43.480000,23.250000,85.889999,68.940002,46.540001,75.430000,120.930000,37.450001,67.269997,45.939999,27.740000,67.379997,50.900002,48.759998,28.620001,18.320000,39.169998,94.190002,124.080002,427.130005,91.989998,22.930000,80.769997,25.030001,87.570000,46.410000,174.139999,18.230000,86.870003,95.440002,94.540001,118.820000,72.830002,53.740002,52.860001,89.519997,69.250000,128.350006,35.630001,53.279999,47.860001,102.6800

## Backtest function

In [4]:
# -----------------------------
# 3) High-vol-only 3-asset vol harvesting backtest
# -----------------------------
def backtest_vol_harvest_3asset_highvol_only(
    prices: pd.DataFrame,
    *,
    tickers: tuple[str, str, str] = ("SPY", "TLT", "GLD"),
    start: str | None = None,
    end: str | None = None,
    rebalance: str = "W-FRI",             # "W-FRI" or "M"
    vol_lookback: int = 60,               # for risk-parity weights + vol targeting
    target_vol: float = 0.10,
    leverage_cap: float = 2.0,

    # High-vol gate
    high_vol_anchor: str = "SPY",
    high_vol_window: int = 63,
    high_vol_threshold: float = 0.20,     # annualized realized vol

    # NEW: drift-based rebalance trigger
    drift_threshold: float | None = 0.08, # L1 weight drift; None disables

    baseline_when_inactive: tuple[float, float, float] = (0.0, 1.0, 0.0),
    cost_bps: float = 1.0,
    periods_per_year: int = 252,
) -> dict:
    """
    High-vol-only volatility harvesting with drift-based rebalancing.
    """
    cols = list(tickers)

    # -----------------------------
    # Prices / returns
    # -----------------------------
    px = prices.copy()
    px.index = pd.to_datetime(px.index).tz_localize(None)
    if start:
        px = px.loc[px.index >= pd.to_datetime(start)]
    if end:
        px = px.loc[px.index <= pd.to_datetime(end)]
    px = px[cols].dropna()

    r = px.pct_change().dropna()
    idx = r.index
    if len(px) < max(vol_lookback, high_vol_window) + 10:
        raise ValueError("Not enough history for requested windows.")

    # -----------------------------
    # High-vol activation signal
    # -----------------------------
    rv = (
        r[high_vol_anchor]
        .rolling(high_vol_window)
        .std(ddof=1) * np.sqrt(periods_per_year)
    )
    active = (rv >= high_vol_threshold).astype(float).reindex(idx).fillna(0.0)

    # -----------------------------
    # Risk-parity weights (shifted)
    # -----------------------------
    vol = r.rolling(vol_lookback).std(ddof=1).shift(1)
    inv = 1.0 / vol.replace(0.0, np.nan)
    w_rp = inv.div(inv.sum(axis=1), axis=0)
    w_rp = w_rp.reindex(idx).fillna(pd.Series([1/3, 1/3, 1/3], index=cols)).ffill()

    # -----------------------------
    # Baseline weights (inactive)
    # -----------------------------
    w_base = pd.DataFrame(
        np.tile(np.array(baseline_when_inactive, dtype=float), (len(idx), 1)),
        index=idx,
        columns=cols,
    )
    w_base = w_base.div(w_base.sum(axis=1), axis=0)

    # -----------------------------
    # Target weights
    # -----------------------------
    cond = (active.values > 0.5).reshape(-1, 1)
    w_target = pd.DataFrame(
        np.where(cond, w_rp.values, w_base.values),
        index=idx,
        columns=cols,
    )

    # -----------------------------
    # Rebalance dates
    # -----------------------------
    rb_dates = px.resample(rebalance).last().index.intersection(idx)
    if len(rb_dates) < 2:
        raise ValueError("Not enough rebalance points.")
    rb_set = set(rb_dates)

    # -----------------------------
    # Constant-mix engine (calendar + drift)
    # -----------------------------
    w_applied = pd.DataFrame(index=idx, columns=cols, dtype=float)
    turnover = pd.Series(0.0, index=idx)
    gross_port = pd.Series(0.0, index=idx)

    vals = pd.Series(w_target.iloc[0].values, index=cols, dtype=float)
    pv = 1.0

    for i, dt in enumerate(idx):
        if i == 0:
            w_applied.loc[dt] = vals.values
            continue

        # evolve sleeves
        vals = vals * (1.0 + r.loc[dt, cols])
        pv_now = float(vals.sum())

        w_now = (vals / pv_now).values
        w_tgt = w_target.loc[dt, cols].values

        # drift (L1 distance)
        drift = float(np.abs(w_now - w_tgt).sum())

        # rebalance condition
        need_reb = (
            (dt in rb_set and active.loc[dt] > 0.5)
            or (drift_threshold is not None and drift >= drift_threshold)
        )

        if need_reb:
            to = drift
            cost = (cost_bps / 1e4) * to * pv_now
            pv_after = max(pv_now - cost, 0.0)
            vals = pd.Series(pv_after * w_tgt, index=cols)
            pv_new = pv_after
            turnover.loc[dt] = to
        else:
            pv_new = pv_now

        gross_port.loc[dt] = (pv_new / pv) - 1.0
        pv = pv_new
        w_applied.loc[dt] = (vals / pv).values if pv > 0 else w_tgt

    # -----------------------------
    # Portfolio vol targeting
    # -----------------------------
    realized = gross_port.rolling(vol_lookback).std(ddof=1) * np.sqrt(periods_per_year)
    lev = (target_vol / realized).clip(0.0, leverage_cap).shift(1).fillna(1.0)

    net = (gross_port * lev).rename("returns").dropna()
    equity = (1.0 + net).cumprod().rename("equity")

    return {
        "returns": net,
        "equity": equity,
        "weights": w_applied.reindex(net.index),
        "turnover": turnover.reindex(net.index),
        "leverage": lev.reindex(net.index),
        "signals": {
            "active": active.reindex(net.index),
            "realized_vol": rv.reindex(net.index),
        },
        "Params": {
            "tickers": tickers,
            "rebalance": rebalance,
            "drift_threshold": drift_threshold,
            "vol_lookback": vol_lookback,
            "target_vol": target_vol,
            "high_vol_window": high_vol_window,
            "high_vol_threshold": high_vol_threshold,
            "baseline_when_inactive": baseline_when_inactive,
            "cost_bps": cost_bps,
        },
    }


## Tearsheet + scoring

In [5]:
# -----------------------------
# 4) Tearsheet + scoring
# -----------------------------
def tearsheet_stats(
    returns: pd.Series,
    *,
    turnover: pd.Series | None = None,
    active: pd.Series | None = None,
    periods_per_year: int = 252,
) -> dict:
    returns = returns.dropna()
    eq = (1.0 + returns).cumprod()
    dd = eq / eq.cummax() - 1.0

    ann_ret = returns.mean() * periods_per_year
    ann_vol = returns.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan

    years = len(eq) / periods_per_year
    cagr = eq.iloc[-1] ** (1.0 / years) - 1.0 if years > 0 else np.nan

    out = {
        "CAGR": cagr,
        "Sharpe": sharpe,
        "AnnualVol": ann_vol,
        "MaxDD": float(dd.min()),
    }
    if turnover is not None:
        out["AvgDailyTurnover"] = float(turnover.mean())
    if active is not None:
        out["ActiveShare"] = float(active.mean())
    return out

def score_triplet(stats: dict) -> float:
    sharpe = float(stats.get("Sharpe", np.nan))
    cagr = float(stats.get("CAGR", np.nan))
    mdd = abs(float(stats.get("MaxDD", np.nan)))
    to = float(stats.get("AvgDailyTurnover", 0.0))
    if not np.isfinite(sharpe) or not np.isfinite(cagr) or not np.isfinite(mdd):
        return -np.inf
    return (1.0 * sharpe) + (2.0 * cagr) - (1.0 * mdd) - (0.5 * to)


## Scan triplets

In [6]:
BT_KWARGS = dict(
    start=START,
    end=END,
    rebalance="W-FRI",
    vol_lookback=60,
    target_vol=0.10,
    leverage_cap=2.0,
    high_vol_window=63,
    high_vol_threshold=0.20,
    baseline_when_inactive=(0.0, 1.0, 0.0),
    cost_bps=1.0,
)

def choose_anchor(sub: pd.DataFrame, triplet: tuple[str,str,str]) -> str:
    # Heuristic: prefer common equity/broad market tickers if present
    prefer = ["SPY","VTI","IVV","VOO","QQQ","IWM","ACWI","VT","EFA","EEM"]
    for p in prefer:
        if p in triplet:
            return p
    # else choose the leg with the highest realized vol (fast and robust)
    r = sub.pct_change().dropna()
    vol = r.std()  # daily std
    return str(vol.idxmax())




fail = 0
new_buffer = []
t0 = time.time()

# -----------------------------
# Scan
# -----------------------------
for i, row in enumerate(triplets_df.itertuples(index=False), start=1):
    a, b, c = row.A, row.B, row.C
    triplet_key = f"{a}-{b}-{c}"

    # Skip if already processed
    if triplet_key in done_keys:
        continue

    sub = px_all[[a, b, c]].dropna()
    if len(sub) < 800:
        continue

    anchor = choose_anchor(sub, (a, b, c))

    try:
        bt = backtest_vol_harvest_3asset_highvol_only(
            sub,
            tickers=(a, b, c),
            high_vol_anchor=anchor,
            **BT_KWARGS
        )

        st = tearsheet_stats(
            bt["returns"],
            turnover=bt["turnover"],
            active=bt["signals"]["active"],
        )

        # attach metadata
        st["Triplet"] = triplet_key
        st["Template"] = getattr(row, "template", None)
        st["HighVolAnchor"] = anchor
        st["Score"] = score_triplet(st)

        # store
        rows.append(st)
        new_buffer.append(st)
        done_keys.add(triplet_key)

        # flush periodically
        if len(new_buffer) >= FLUSH_EVERY:
            pd.DataFrame(new_buffer).to_csv(
                CHECKPOINT_PATH,
                mode="a",
                header=not os.path.exists(CHECKPOINT_PATH),
                index=False
            )
            new_buffer = []

        if i % PRINT_EVERY == 0:
            elapsed = time.time() - t0
            print(f"[progress] iter={i} done={len(done_keys)} fail={fail} elapsed={elapsed:.1f}s")

    except Exception:
        fail += 1

# Final flush
if new_buffer:
    pd.DataFrame(new_buffer).to_csv(
        CHECKPOINT_PATH,
        mode="a",
        header=not os.path.exists(CHECKPOINT_PATH),
        index=False
    )

print(f"Finished. done={len(done_keys)} fail={fail}")




out = pd.DataFrame(rows)
out = out.sort_values("Score", ascending=False)
out.head(20)


[progress] iter=21500 done=21500 fail=0 elapsed=1590.8s
[progress] iter=22000 done=22000 fail=0 elapsed=3086.1s
[progress] iter=22500 done=22500 fail=0 elapsed=4347.8s
[progress] iter=23000 done=22550 fail=0 elapsed=4542.6s
[progress] iter=23500 done=23050 fail=0 elapsed=6223.2s
[progress] iter=24000 done=23550 fail=0 elapsed=7805.3s
[progress] iter=24500 done=24050 fail=0 elapsed=9190.2s
[progress] iter=25000 done=24550 fail=0 elapsed=10351.9s
[progress] iter=25500 done=25050 fail=0 elapsed=11915.2s
[progress] iter=26000 done=25550 fail=0 elapsed=13504.2s
[progress] iter=26500 done=26050 fail=0 elapsed=15077.4s
[progress] iter=27000 done=26550 fail=0 elapsed=16665.6s
[progress] iter=27500 done=27050 fail=0 elapsed=18275.4s
[progress] iter=28000 done=27550 fail=0 elapsed=19875.4s
[progress] iter=28500 done=28050 fail=0 elapsed=21447.0s
[progress] iter=29000 done=28550 fail=0 elapsed=23029.6s
[progress] iter=29500 done=29050 fail=0 elapsed=24556.2s
[progress] iter=30000 done=29550 fail=

,CAGR,Sharpe,AnnualVol,MaxDD,AvgDailyTurnover,ActiveShare,Triplet,Template,HighVolAnchor,Score
14454,0.090813,16.664648,0.005218,-0.001865,0.000259,0.139214,JEPQ-SGOV-SGOL,equity_bond_commodity,SGOL,16.844278
14458,0.090805,16.662568,0.005218,-0.001851,0.000259,0.138151,JEPQ-SGOV-AAAU,equity_bond_commodity,AAAU,16.842197
14447,0.090794,16.657344,0.005219,-0.001887,0.000258,0.139214,JEPQ-SGOV-GLD,equity_bond_commodity,GLD,16.836915
14456,0.090807,16.656546,0.005220,-0.001886,0.000259,0.139214,JEPQ-SGOV-GLDM,equity_bond_commodity,GLDM,16.836143
14459,0.090554,16.618695,0.005218,-0.001858,0.000290,0.140276,JEPQ-SGOV-IAUM,equity_bond_commodity,IAUM,16.797799
14450,0.090543,16.606948,0.005221,-0.001868,0.000289,0.140276,JEPQ-SGOV-IAU,equity_bond_commodity,IAU,16.786021
14452,0.088936,16.232996,0.005250,-0.001467,0.000228,0.157279,JEPQ-SGOV-PDBC,equity_bond_commodity,PDBC,16.409287
20304,0.087547,16.174531,0.005190,-0.001923,0.000214,0.132457,CGDV-SGOV-SGOL,equity_bond_commodity,SGOL,16.347595
20308,0.087540,16.174246,0.005190,-0.001909,0.000214,0.131446,CGDV-SGOV-AAAU,equity_bond_commodity,AAAU,16.347310
20297,0.087529,16.167704,0.005192,-0.001945,0.000213,0.132457,CGDV-SGOV-GLD,equity_bond_commodity,GLD,16.340710


## Inspect top candidates

In [7]:
# -----------------------------
# 6) Inspect top candidates + plots
# -----------------------------
TOP_N = 10
top = out.head(TOP_N).copy()

# display with light formatting
show = top.copy()
for col in ["CAGR","AnnualVol","MaxDD","AvgDailyTurnover","ActiveShare","Sharpe","Score"]:
    if col in show.columns:
        show[col] = show[col].astype(float)

show


,CAGR,Sharpe,AnnualVol,MaxDD,AvgDailyTurnover,ActiveShare,Triplet,Template,HighVolAnchor,Score
14454,0.090813,16.664648,0.005218,-0.001865,0.000259,0.139214,JEPQ-SGOV-SGOL,equity_bond_commodity,SGOL,16.844278
14458,0.090805,16.662568,0.005218,-0.001851,0.000259,0.138151,JEPQ-SGOV-AAAU,equity_bond_commodity,AAAU,16.842197
14447,0.090794,16.657344,0.005219,-0.001887,0.000258,0.139214,JEPQ-SGOV-GLD,equity_bond_commodity,GLD,16.836915
14456,0.090807,16.656546,0.005220,-0.001886,0.000259,0.139214,JEPQ-SGOV-GLDM,equity_bond_commodity,GLDM,16.836143
14459,0.090554,16.618695,0.005218,-0.001858,0.000290,0.140276,JEPQ-SGOV-IAUM,equity_bond_commodity,IAUM,16.797799
14450,0.090543,16.606948,0.005221,-0.001868,0.000289,0.140276,JEPQ-SGOV-IAU,equity_bond_commodity,IAU,16.786021
14452,0.088936,16.232996,0.005250,-0.001467,0.000228,0.157279,JEPQ-SGOV-PDBC,equity_bond_commodity,PDBC,16.409287
20304,0.087547,16.174531,0.005190,-0.001923,0.000214,0.132457,CGDV-SGOV-SGOL,equity_bond_commodity,SGOL,16.347595
20308,0.087540,16.174246,0.005190,-0.001909,0.000214,0.131446,CGDV-SGOV-AAAU,equity_bond_commodity,AAAU,16.347310
20297,0.087529,16.167704,0.005192,-0.001945,0.000213,0.132457,CGDV-SGOV-GLD,equity_bond_commodity,GLD,16.340710
